# Portfolio Training with Walk-Forward Validation

Train PPO agent for portfolio rebalancing using walk-forward validation.

**Configuration:** All parameters defined in `config.py`

**Key Features:**
- Walk-forward validation (no data leakage)
- Configurable reward functions
- 7 tech stocks, weekly rebalancing
- 149 features (macro + technical + volume + risk)

## Setup

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import torch.nn as nn
import matplotlib.pyplot as plt

from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Import config and RL system
from config import *
from rl_system import (
    PortfolioEnv,
    PortfolioMonitor,
    TrainingLogger,
    create_walk_forward_folds,
    print_fold_summary
)

print("✓ Imports complete")

## Configuration

In [ ]:
# Print current configuration
print_config()

## Load Data

In [ ]:
# Load training data using config
train_data = load_data()

print(f"\n✓ Loaded {len(train_data):,} samples")
print(f"  Date range: {train_data[DATE_COL].min().date()} to {train_data[DATE_COL].max().date()}")
print(f"  Tickers: {train_data['ticker'].nunique()}")
print(f"  Features: {len(FEATURE_COLS)}")

## Create Walk-Forward Folds

In [ ]:
# Create folds using config parameters
folds = create_walk_forward_folds(
    train_data=train_data,
    n_folds=N_FOLDS,
    min_train_size=MIN_TRAIN_SIZE,
    date_col=DATE_COL
)

print_fold_summary(folds)

## Training Function

In [ ]:
def train_fold(fold_idx, train_data, val_data):
    """
    Train PPO agent on one fold.
    All parameters from config.py.
    """
    print(f"\n{'='*80}")
    print(f"FOLD {fold_idx + 1}/{N_FOLDS}")
    print(f"{'='*80}")
    
    # Verify data
    print(f"\n📊 Data:")
    print(f"   Train: {train_data[DATE_COL].min().date()} to {train_data[DATE_COL].max().date()} ({len(train_data['date'].unique())} periods)")
    print(f"   Val:   {val_data[DATE_COL].min().date()} to {val_data[DATE_COL].max().date()} ({len(val_data['date'].unique())} periods)")
    
    assert train_data[DATE_COL].max() < val_data[DATE_COL].min(), "Data leakage!"
    print(f"   ✓ No data leakage")
    
    # Create environments
    def make_env(data):
        env = PortfolioEnv(
            data=data.copy(),
            feature_cols=FEATURE_COLS,
            tickers=TICKERS,
            rebalance_frequency=REBALANCE_FREQUENCY,
            transaction_cost=TRANSACTION_COST,
            max_weight_per_asset=MAX_WEIGHT_PER_ASSET,
            reward_lookback=REWARD_LOOKBACK,
            initial_capital=INITIAL_CAPITAL,
            reward_type=REWARD_TYPE,
            seed=RANDOM_SEED + fold_idx
        )
        return PortfolioMonitor(env)
    
    train_env = DummyVecEnv([lambda: make_env(train_data)])
    val_env = DummyVecEnv([lambda: make_env(val_data)])
    
    # Setup output directory
    fold_dir = Path(OUTPUT_DIR) / f"fold_{fold_idx}"
    fold_dir.mkdir(parents=True, exist_ok=True)
    
    # Create callback
    logger = TrainingLogger(
        eval_env=val_env,
        eval_freq=EVAL_FREQ,
        fold_dir=fold_dir,
        patience=20,
        verbose=VERBOSE
    )
    
    # Create model
    model = PPO(
        "MlpPolicy",
        train_env,
        policy_kwargs=POLICY_KWARGS,
        learning_rate=LEARNING_RATE,
        n_steps=N_STEPS,
        batch_size=BATCH_SIZE,
        n_epochs=N_EPOCHS,
        gamma=GAMMA,
        gae_lambda=GAE_LAMBDA,
        clip_range=CLIP_RANGE,
        ent_coef=ENT_COEF,
        vf_coef=VF_COEF,
        max_grad_norm=MAX_GRAD_NORM,
        verbose=0,
        tensorboard_log=str(fold_dir / "tensorboard") if USE_TENSORBOARD else None
    )
    
    # Train
    print(f"\n🤖 Training for {TOTAL_TIMESTEPS:,} timesteps...\n")
    model.learn(
        total_timesteps=TOTAL_TIMESTEPS,
        callback=logger,
        progress_bar=False
    )
    
    # Save model
    if SAVE_ALL_MODELS:
        model.save(str(fold_dir / "final_model"))
    
    # Final validation
    print(f"\n📈 Final Validation:")
    obs = val_env.reset()
    done = False
    
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, done, info = val_env.step(action)
        
        if done[0]:
            stats = info[0].get('portfolio_stats', {})
    
    # Print results
    print(f"   Total Return:  {stats['total_return']*100:>7.2f}%")
    print(f"   Sharpe Ratio:  {stats['sharpe_ratio']:>7.3f}")
    print(f"   Volatility:    {stats['volatility']*100:>7.2f}%")
    print(f"   Max Drawdown:  {stats['max_drawdown']*100:>7.2f}%")
    print(f"   Final Value:   ${stats['portfolio_value']:>,.2f}")
    print(f"   Best Val Sharpe: {logger.best_val_sharpe:>7.3f}")
    
    # Return results
    return {
        'fold': fold_idx,
        'total_return': stats['total_return'],
        'sharpe_ratio': stats['sharpe_ratio'],
        'volatility': stats['volatility'],
        'max_drawdown': stats['max_drawdown'],
        'portfolio_value': stats['portfolio_value'],
        'best_val_sharpe': logger.best_val_sharpe,
        'early_stopped': logger.early_stopped,
        'model_path': str(fold_dir / "best_model")
    }

print("✓ Training function defined")

## Train All Folds

In [ ]:
# Train all folds
results = []

for fold in folds:
    result = train_fold(
        fold_idx=fold['fold_idx'],
        train_data=fold['train'],
        val_data=fold['val']
    )
    results.append(result)

print(f"\n{'='*80}")
print("✓ ALL FOLDS COMPLETE")
print(f"{'='*80}")

## Results Summary

In [ ]:
# Create results DataFrame
results_df = pd.DataFrame(results)

# Save results
output_path = Path(OUTPUT_DIR) / 'fold_results.csv'
results_df.to_csv(output_path, index=False)

# Print summary
print("\n" + "="*80)
print("RESULTS SUMMARY")
print("="*80)

print(f"\n📊 Validation Performance:")
print(f"   Sharpe Ratio:  {results_df['sharpe_ratio'].mean():.3f} ± {results_df['sharpe_ratio'].std():.3f}")
print(f"   Total Return:  {results_df['total_return'].mean():.2%} ± {results_df['total_return'].std():.2%}")
print(f"   Volatility:    {results_df['volatility'].mean():.2%} ± {results_df['volatility'].std():.2%}")
print(f"   Max Drawdown:  {results_df['max_drawdown'].mean():.2%} ± {results_df['max_drawdown'].std():.2%}")
print(f"   Best Sharpe:   {results_df['best_val_sharpe'].mean():.3f} ± {results_df['best_val_sharpe'].std():.3f}")

print(f"\n💾 Saved to: {output_path}")

# Display table
results_df

## Visualize Results

In [ ]:
# Create visualizations
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Sharpe Ratio
axes[0, 0].bar(results_df['fold'], results_df['sharpe_ratio'], color='steelblue')
axes[0, 0].axhline(results_df['sharpe_ratio'].mean(), color='red', linestyle='--', label='Mean')
axes[0, 0].set_xlabel('Fold')
axes[0, 0].set_ylabel('Sharpe Ratio')
axes[0, 0].set_title('Sharpe Ratio by Fold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Total Return
axes[0, 1].bar(results_df['fold'], results_df['total_return']*100, color='green')
axes[0, 1].axhline(results_df['total_return'].mean()*100, color='red', linestyle='--', label='Mean')
axes[0, 1].set_xlabel('Fold')
axes[0, 1].set_ylabel('Total Return (%)')
axes[0, 1].set_title('Total Return by Fold')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Max Drawdown
axes[1, 0].bar(results_df['fold'], results_df['max_drawdown']*100, color='coral')
axes[1, 0].axhline(results_df['max_drawdown'].mean()*100, color='red', linestyle='--', label='Mean')
axes[1, 0].set_xlabel('Fold')
axes[1, 0].set_ylabel('Max Drawdown (%)')
axes[1, 0].set_title('Max Drawdown by Fold')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Volatility
axes[1, 1].bar(results_df['fold'], results_df['volatility']*100, color='purple')
axes[1, 1].axhline(results_df['volatility'].mean()*100, color='red', linestyle='--', label='Mean')
axes[1, 1].set_xlabel('Fold')
axes[1, 1].set_ylabel('Volatility (%)')
axes[1, 1].set_title('Volatility by Fold')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()

# Save plot
plot_path = Path(OUTPUT_DIR) / 'fold_results.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()

print(f"\n💾 Plot saved to: {plot_path}")

## Notes

- All parameters defined in `config.py`
- Test data kept separate for final evaluation
- Models saved to `models/` directory
- Best model per fold saved with `TrainingLogger`